# MobileRec: Popularity and ItemKNN Baselines

Popularity and ItemKNN serve as non-neural baselines for comparison with the neural models. Popularity recommends the same globally frequent apps to every user. ItemKNN is a personalized collaborative filtering model that combines item similarity with each user’s interaction history.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from time import perf_counter
import duckdb
from scipy.sparse import coo_matrix, load_npz, save_npz
from sklearn.neighbors import NearestNeighbors

In [2]:
run_name = "all_users"
seed = 42
top_k = 10

The `top_k` variable specifies the length of the recommendation list evaluated for each user. With `top_k = 10`, evaluation checks whether the actual next app appears among the model’s ten highest-ranked recommendations.

In [3]:
split_dir = Path("artifacts") / "split_all_users"
results_dir = Path("artifacts") / "baselines_all_users"
temp_dir = Path("artifacts") / "baseline_temp_all_users"
metadata_path = Path("artifacts") / "final" / "metadata_final.parquet"

train_path = split_dir / "train_all_users.parquet"
valid_path = split_dir / "validation_all_users.parquet"
test_path = split_dir / "test_all_users.parquet"
user_map_path = split_dir / "users_map_all_users.parquet"
item_map_path = split_dir / "items_map_all_users.parquet"
pop_path = split_dir / "apps_popularity_all_users.parquet"
coverage_path = split_dir / "start_coverage_all_users.parquet"

results_dir.mkdir(parents=True, exist_ok=True)
temp_dir.mkdir(exist_ok=True)

required_paths = [train_path, valid_path, test_path, user_map_path, item_map_path, pop_path, coverage_path, metadata_path]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f"File does not exist: {path}")

In [4]:
user_map = pd.read_parquet(user_map_path).sort_values("user_idx")
item_map = pd.read_parquet(item_map_path).sort_values("item_idx")
app_counts = pd.read_parquet(pop_path)
coverage_data = pd.read_parquet(coverage_path).sort_values("split_name")

In [5]:
num_users = len(user_map)
num_items = len(item_map)

def load_targets(split_path):
    targets = pd.read_parquet(split_path, columns=["uid", "app_package"])
    targets = targets.merge(user_map[["uid", "user_idx"]], on="uid", how="inner")
    targets = targets.merge(item_map[["app_package", "item_idx"]], on="app_package", how="inner")
    targets["user_idx"] = targets["user_idx"].astype("int32")
    targets["item_idx"] = targets["item_idx"].astype("int32")
    return targets.sort_values("user_idx").reset_index(drop=True)

valid_targets = load_targets(valid_path)
test_targets = load_targets(test_path)

print("Users in mapping:", num_users)
print("Candidate applications:", num_items)
print("Validation targets for evaluation:", len(valid_targets))
print("Test targets for evaluation:", len(test_targets))
coverage_data

Users in mapping: 700111
Candidate applications: 10172
Validation targets for evaluation: 700108
Test targets for evaluation: 700096


,split_name,all_targets,known_items,cold_items,coverage_percent
0,test,700111,700096.0,15.0,99.997857
1,validation,700111,700108.0,3.0,99.999571


The files produced by the chronological split are loaded. The textual identifiers `uid` and `app_package` are linked to the numeric indices `user_idx` and `item_idx` used by the interaction matrix and models.

The mappings are derived exclusively from training data and cover 700,111 users and 10,172 apps. One of the original 10,173 apps has no training interaction. Its three validation and 15 test targets are retained in the split files but excluded from standard ranking metrics.

## Popularity Model

Popularity is a simple, non-personalized model. App frequencies are computed exclusively from the training set, and apps are ranked in descending order of interaction count. Every user receives the same ten most popular apps.

The model does not use individual user histories and has no hyperparameters to tune. It provides a baseline for assessing performance based on global popularity alone.

In [6]:
def eval_top_k(targets, rec):
    target_items = targets["item_idx"].to_numpy()
    matches = rec == target_items[:, None]
    hits = matches.any(axis=1)
    ranks = matches.argmax(axis=1)
    ndcg = 0.0
    
    if hits.any():
        ndcg = (1 / np.log2(ranks[hits] + 2)).sum() / len(targets)
    return {
        "hit_recall_at_10": float(hits.mean()),
        "ndcg_at_10": float(ndcg),
        "evaluated_users": len(targets)
    }

The `eval_top_k` function compares each user’s target app with their ten recommendations. A target found in the list receives a Hit@10 and Recall@10 value of one; otherwise, both are zero. These metrics are equivalent here because each user has exactly one target. NDCG@10 additionally accounts for the target’s rank, giving greater weight to hits near the top of the list.

Only targets whose apps appear in the training mapping are evaluated. The same rules apply to both baselines and the subsequent neural models.

In [7]:
start_time = perf_counter()
app_counts = app_counts.sort_values(["interaction_count", "item_idx"], ascending=[False, True])
top_items = app_counts["item_idx"].head(top_k).to_numpy()
valid_recs = np.tile(top_items, (len(valid_targets), 1))
valid_eval = eval_top_k(valid_targets, valid_recs)
test_recs = np.tile(top_items, (len(test_targets), 1))
test_eval = eval_top_k(test_targets, test_recs)
model_time = perf_counter() - start_time

print("Popularity validation:", valid_eval)
print("Popularity test:", test_eval)
print(f"Popularity time: {model_time:.2f} sekundi")

Popularity validation: {'hit_recall_at_10': 0.012480931513423643, 'ndcg_at_10': 0.006410941517227603, 'evaluated_users': 700108}
Popularity test: {'hit_recall_at_10': 0.01505079303409818, 'ndcg_at_10': 0.0077618861950043035, 'evaluated_users': 700096}
Popularity time: 0.23 sekundi


In [8]:
metadata_df = pd.read_parquet(metadata_path, columns=["app_package", "app_name"])
top_apps = app_counts.head(top_k).merge(metadata_df, on="app_package", how="left")
top_apps[["app_name", "app_package", "interaction_count", "user_count"]]

,app_name,app_package,interaction_count,user_count
0,Snapchat,com.snapchat.android,10940,10550
1,Facebook Lite,com.facebook.lite,10718,10380
2,Candy Crush Saga,com.king.candycrushsaga,10232,9954
3,CapCut - Video Editor,com.lemon.lvoverseas,9460,9184
4,TikTok,com.zhiliaoapp.musically,9394,9134
5,Candy Crush Soda Saga,com.king.candycrushsodasaga,9173,8961
6,Bigo Live - Live Streaming App,sg.bigo.live,8924,8745
7,ZOOM Cloud Meetings,us.zoom.videomeetings,8579,8368
8,Cash App,com.squareup.cash,8512,8278
9,Twitch: Live Game Streaming,tv.twitch.android.app,8449,8262


The most popular training apps are Snapchat, Facebook Lite, Candy Crush Saga, CapCut, and TikTok. The list reflects global interaction frequency rather than individual user histories.

Popularity achieves validation Hit/Recall@10 of 1.2481% and NDCG@10 of 0.6411%. Test Hit/Recall@10 is 1.5051%, and NDCG@10 is 0.7762%. These are ranking metrics rather than conventional classification accuracy: the model selects only ten recommendations from 10,172 candidates.

## ItemKNN Model

ItemKNN is a personalized collaborative filtering model based on the k nearest neighbors of each app. It uses the binary training interaction matrix to compute cosine similarity between apps. Higher similarity indicates that two apps tend to appear in the histories of the same users. Each candidate’s score is the sum of its similarities to apps in the target user’s training history.

ItemKNN uses the same interaction signal as the neural models while generating recommendations without a neural network. The structure of MobileRec also supports choosing ItemKNN over UserKNN: the median is only eight interactions per user but 1,411 per app, providing more observations for comparing apps than short user histories.

In [9]:
train_indices_path = results_dir / f"train_indices_{run_name}_seed_{seed}.parquet"
matrix_path = results_dir / f"user_item_matrix_{run_name}_seed_{seed}.npz"

if train_indices_path.exists():
    print("Numeric training pairs already exist.")
else:
    start_time = perf_counter()
    con = duckdb.connect()
    con.execute("SET memory_limit = '2GB'")
    con.execute(f"SET temp_directory = '{temp_dir.as_posix()}'")
    con.execute(f"""COPY (
        SELECT DISTINCT user_map.user_idx, item_map.item_idx
        FROM read_parquet('{train_path.as_posix()}') AS train_data
        INNER JOIN read_parquet('{user_map_path.as_posix()}') AS user_map USING (uid)
        INNER JOIN read_parquet('{item_map_path.as_posix()}') AS item_map USING (app_package)
    ) TO '{train_indices_path.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)""")
    con.close()
    print(f"Numericki train parovi su napravljeni za {(perf_counter()-start_time)/60:.2f} minutes")

if matrix_path.exists():
    user_item_matrix = load_npz(matrix_path).tocsr()
    print("Sparse matrix already exists.")
else:
    train_indices = pd.read_parquet(train_indices_path)
    user_item_matrix = coo_matrix(
        (np.ones(len(train_indices), dtype=np.float32),
        (train_indices["user_idx"].to_numpy(), train_indices["item_idx"].to_numpy())),
        shape=(num_users, num_items), dtype=np.float32).tocsr()
    user_item_matrix.data[:] = 1
    save_npz(matrix_path, user_item_matrix)
    del train_indices
    print("Sparse matrica je sacuvana.")

matrix_memory = (user_item_matrix.data.nbytes + user_item_matrix.indices.nbytes
                 + user_item_matrix.indptr.nbytes) / 1024**2
matrix_density = user_item_matrix.nnz / (num_users * num_items) * 100

print("Dimenzije matrice:", user_item_matrix.shape)
print("Jedinstveni user-app parovi:", user_item_matrix.nnz)
print(f"Gustina matrice: {matrix_density:.4f}%")
print(f"Memorija sparse matrice: {matrix_memory:.2f} MiB")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Numericki train parovi su napravljeni za 0.05 minutes
Sparse matrica je sacuvana.
Dimenzije matrice: (700111, 10172)
Jedinstveni user-app parovi: 17748311
Gustina matrice: 0.2492%
Memorija sparse matrice: 138.08 MiB


Sparse matrica vec postoji.
Dimenzije matrice: (700111, 10172)
Jedinstveni user-app parovi: 17748311
Gustina matrice: 0.2492%
Memorija sparse matrice: 138.08 MiB


The binary sparse matrix has 700,111 user rows and 10,172 app columns. A value of one indicates at least one training interaction for that user–app pair.

The matrix contains 17,748,311 unique pairs, compared with 17,896,797 training rows, because repeated interactions for the same user and app are collapsed into a single binary value. Only 0.2492% of all possible pairs are observed. Storing only nonzero entries reduces memory usage to approximately 138.08 MiB. The same matrix is reused in subsequent notebooks.

In [10]:
max_k = 300
k_options = [100, 200, 300]
batch_size = 250

In [11]:
knn_path = results_dir / f"item_neighbors_{max_k}_{run_name}_seed_{seed}.npz"

if knn_path.exists():
    knn_data = np.load(knn_path)
    knn_rows = knn_data["rows"]
    knn_cols = knn_data["cols"]
    knn_scores = knn_data["scores"]
    knn_ranks = knn_data["ranks"]
    knn_data.close()
    knn_build_time = np.nan
    print("ItemKNN neighbors already exist.")
else:
    start_time = perf_counter()
    item_user_matrix = user_item_matrix.T.tocsr()
    knn = NearestNeighbors(n_neighbors=max_k + 1, metric="cosine", algorithm="brute", n_jobs=1)
    knn.fit(item_user_matrix)
    distances, indices = knn.kneighbors(item_user_matrix, return_distance=True)
    rows = []
    cols = []
    scores = []
    ranks = []

    for item_idx in range(num_items):
        mask = indices[item_idx] != item_idx
        c_cols = indices[item_idx][mask][:max_k]
        c_scores = (1 - distances[item_idx][mask])[:max_k]
        c_ranks = np.arange(1, len(c_cols) + 1)
        rows.append(np.full(len(c_cols), item_idx, dtype=np.int32))
        cols.append(c_cols.astype(np.int32))
        scores.append(c_scores.astype(np.float32))
        ranks.append(c_ranks.astype(np.int16))

    knn_rows = np.concatenate(rows)
    knn_cols = np.concatenate(cols)
    knn_scores = np.concatenate(scores)
    knn_ranks = np.concatenate(ranks)

    np.savez_compressed(knn_path,rows=knn_rows, cols=knn_cols, scores=knn_scores,ranks=knn_ranks )
    knn_build_time = perf_counter() - start_time
    print(f"ItemKNN susedi su napravljeni za {knn_build_time/60:.2f} minutes")

print("Saved application relationships:", len(knn_scores))

ItemKNN susedi su napravljeni za 0.61 minutes
Saved application relationships: 3051600


In [12]:
def make_sim(k):
    mask = knn_ranks <= k
    sim = coo_matrix((knn_scores[mask], (knn_rows[mask], knn_cols[mask])),shape=(num_items, num_items), dtype=np.float32).tocsr()
    sim.eliminate_zeros()
    return sim

def knn_recs(history_matrix, sim, targets):
    recs = np.empty((len(targets), top_k), dtype=np.int32)
    users = targets["user_idx"].to_numpy()

    for start in range(0, len(targets), batch_size):
        end = min(start + batch_size, len(targets))
        scores = (history_matrix[users[start:end]] @ sim).toarray()
        recs[start:end] = np.argsort(-scores, axis=1, kind="stable")[:, :top_k]

    return recs


In [13]:
tuning_rows = []

for k in k_options:
    sim = make_sim(k)
    start_time = perf_counter()
    valid_recs = knn_recs(user_item_matrix, sim, valid_targets)
    curr_eval = eval_top_k(valid_targets, valid_recs)

    tuning_rows.append({
        "k": k,
        "validation_hit_recall_10": curr_eval["hit_recall_at_10"],
        "validation_ndcg_10": curr_eval["ndcg_at_10"],
        "evaluation_seconds": perf_counter() - start_time})

    print(f"K: {k} | Hit@10: {curr_eval['hit_recall_at_10']:.6f} | NDCG@10: {curr_eval['ndcg_at_10']:.6f}")
    del sim, valid_recs

tuning_results = pd.DataFrame(tuning_rows)
tuning_results = tuning_results.sort_values(["validation_ndcg_10", "k"], ascending=[False, True]).reset_index(drop=True)
selected_k = int(tuning_results.loc[0, "k"])
tuning_path = results_dir / f"itemknn_tuning_{run_name}_seed_{seed}.csv"
tuning_results.to_csv(tuning_path, index=False)

print("Izabrani k:", selected_k)
tuning_results


K: 100 | Hit@10: 0.013409 | NDCG@10: 0.006674
K: 200 | Hit@10: 0.013441 | NDCG@10: 0.006680
K: 300 | Hit@10: 0.013395 | NDCG@10: 0.006668
Izabrani k: 200


,k,validation_hit_recall_10,validation_ndcg_10,evaluation_seconds
0,200,0.013441,0.006680,409.834353
1,100,0.013409,0.006674,321.409985
2,300,0.013395,0.006668,472.904098


K: 300 | Hit@10: 0.013395 | NDCG@10: 0.006668
Izabrani k: 200


,k,validation_hit_recall_10,validation_ndcg_10,evaluation_seconds
0,200,0.013441,0.006680,142.875043
1,100,0.013409,0.006674,133.874350
2,300,0.013395,0.006668,168.380684


Up to 300 neighbors per app are computed using cosine similarity derived only from training data. Values of `k = 100`, `k = 200`, and `k = 300` are evaluated on the validation set. The highest NDCG@10, 0.006680, is obtained with `k = 200`, which is selected for testing.

Here, `k` is the number of neighboring apps used for scoring; it is distinct from `top_k = 10`, the length of the output recommendation list. Recommendations are computed in batches of 250 users to limit RAM usage without changing the results.

In [16]:
selected_sim = make_sim(selected_k)

start_time = perf_counter()
knn_test_recs = knn_recs(user_item_matrix, selected_sim, test_targets)
knn_test_eval = eval_top_k(test_targets, knn_test_recs)
knn_test_time = perf_counter() - start_time

print("ItemKNN test:", knn_test_eval)
print(f"ItemKNN test time: {knn_test_time:.2f} seconds")


ItemKNN test: {'hit_recall_at_10': 0.016871971843861414, 'ndcg_at_10': 0.00825634296347505, 'evaluated_users': 700096}
ItemKNN test time: 273.44 seconds


In [17]:
valid_coverage = coverage_data.loc[coverage_data["split_name"] == "validation", "coverage_percent"].iloc[0]
test_coverage = coverage_data.loc[coverage_data["split_name"] == "test", "coverage_percent"].iloc[0]

baseline_results = pd.DataFrame([
    {
        "model": "Popularity",
        "users": num_users,
        "seed": seed,
        "validation_Hit@10": valid_eval["hit_recall_at_10"],
        "validation_Recall@10": valid_eval["hit_recall_at_10"],
        "validation_NDCG@10": valid_eval["ndcg_at_10"],
        "test_Hit@10": test_eval["hit_recall_at_10"],
        "test_Recall@10": test_eval["hit_recall_at_10"],
        "test_NDCG@10": test_eval["ndcg_at_10"],
        "test_coverage": test_coverage,
        "model_time_seconds": model_time,
        "notes": "Globalna popularnost iz train skupa"
    },
    {
        "model": "ItemKNN",
        "users": num_users,
        "seed": seed,
        "validation_Hit@10": tuning_results.loc[0, "validation_hit_recall_10"],
        "validation_Recall@10": tuning_results.loc[0, "validation_hit_recall_10"],
        "validation_NDCG@10": tuning_results.loc[0, "validation_ndcg_10"],
        "test_Hit@10": knn_test_eval["hit_recall_at_10"],
        "test_Recall@10": knn_test_eval["hit_recall_at_10"],
        "test_NDCG@10": knn_test_eval["ndcg_at_10"],
        "test_coverage": test_coverage,
        "model_time_seconds": knn_build_time,
        "notes": f"Binarni ItemKNN sa k = {selected_k}"
    }
])

baseline_results = baseline_results.sort_values("test_NDCG@10", ascending=False).reset_index(drop=True)

results_path = results_dir / f"baseline_results_{run_name}_seed_{seed}.csv"
baseline_results.to_csv(results_path, index=False)

print("Saved results:", results_path)
baseline_results.round(6)


Saved results: artifacts\baselines_all_users\baseline_results_all_users_seed_42.csv


,model,users,seed,validation_Hit@10,validation_Recall@10,validation_NDCG@10,test_Hit@10,test_Recall@10,test_NDCG@10,test_coverage,model_time_seconds,notes
0,ItemKNN,700111,42,0.013441,0.013441,0.006680,0.016872,0.016872,0.008256,99.997857,36.727301,Binarni ItemKNN sa k = 200
1,Popularity,700111,42,0.012481,0.012481,0.006411,0.015051,0.015051,0.007762,99.997857,0.227138,Globalna popularnost iz train skupa


On the test set, ItemKNN with `k = 200` achieves Hit/Recall@10 of 1.6872% and NDCG@10 of 0.8256%. Popularity achieves 1.5051% and 0.7762%, respectively, on the same targets.

ItemKNN performs better on both metrics, indicating that individual histories and app similarities provide additional value beyond a shared list of globally popular apps. Popularity nevertheless remains a strong baseline because interactions are concentrated among popular apps.